# R7 — Hotline KB via **dedicated** OGX + pgvector

Targets the stack you **installed** (not the Playground):
- OGX: `hotline-rag-ogx-service:8321`
- DB: `hotline-rag-pgvector`
- LLM: same Granite vLLM in the project

Install first: `manifests/hotline-rag/` + [`README-GENAI-RAG.md`](../README-GENAI-RAG.md) §R7.

**Before running:** clone the lab repo (or copy `data/hotline-kb/upload/`) into the workbench.

In [ ]:
%pip install -q requests

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import requests

# Dedicated Hotline RAG OGX (installed in R7) — no trailing /v1 on base URL
OGX = "http://hotline-rag-ogx-service.genai-hotline.svc.cluster.local:8321"
V1 = f"{OGX}/v1"

KB_DIR = Path("data/hotline-kb/upload")
if not KB_DIR.is_dir():
    KB_DIR = Path("../data/hotline-kb/upload")
if not KB_DIR.is_dir():
    KB_DIR = Path("upload")

print("OGX:", OGX)
print("KB_DIR:", KB_DIR.resolve(), "exists=" + str(KB_DIR.is_dir()))

In [ ]:
models = requests.get(f"{V1}/models", timeout=60).json()["data"]
for m in models:
    mt = (m.get("custom_metadata") or {}).get("model_type")
    print(f"{mt:10} {m['id']}")

llm_id = next(
    m["id"]
    for m in models
    if (m.get("custom_metadata") or {}).get("model_type") == "llm"
)
emb = next(
    (
        m
        for m in models
        if (m.get("custom_metadata") or {}).get("model_type") == "embedding"
        and "granite-embedding" in m["id"]
    ),
    None,
)
if emb is None:
    emb = next(
        (
            m
            for m in models
            if (m.get("custom_metadata") or {}).get("model_type") == "embedding"
        ),
        None,
    )
assert emb is not None, (
    "No embedding model on this OGXServer. Apply 04-ogx-config.yaml "
    "+ re-apply 03-ogxserver.yaml, wait Ready, re-run."
)
emb_id = emb["id"]
emb_dim = int((emb.get("custom_metadata") or {}).get("embedding_dimension", 768))
print("LLM:", llm_id)
print("EMB:", emb_id, "dim", emb_dim)


In [ ]:
vs = requests.post(
    f"{V1}/vector_stores",
    json={
        "name": "hotline-kb-pgvector",
        "extra_body": {
            "embedding_model": emb_id,
            "embedding_dimension": emb_dim,
            "provider_id": "pgvector",
        },
    },
    timeout=120,
).json()
vs_id = vs["id"]
print(json.dumps(vs, indent=2))
print("vector_store_id =", vs_id)

In [ ]:
assert KB_DIR.is_dir(), f"Missing KB folder: {KB_DIR} — clone the lab repo or copy data/hotline-kb/upload here"

file_ids = []
for path in sorted(KB_DIR.glob("*")):
    if path.suffix.lower() not in {".txt", ".csv", ".md"}:
        continue
    with path.open("rb") as fh:
        r = requests.post(
            f"{V1}/files",
            files={"file": (path.name, fh, "text/plain")},
            data={"purpose": "assistants"},
            timeout=120,
        )
    r.raise_for_status()
    info = r.json()
    file_ids.append((path.name, info["id"]))
    print("uploaded", path.name, "->", info["id"])

for name, fid in file_ids:
    r = requests.post(
        f"{V1}/vector_stores/{vs_id}/files",
        json={
            "file_id": fid,
            "chunking_strategy": {
                "type": "static",
                "static": {"max_chunk_size_tokens": 800, "chunk_overlap_tokens": 100},
            },
        },
        timeout=300,
    )
    r.raise_for_status()
    print("indexed", name, r.json().get("status"))

In [ ]:
SYSTEM = """You are Hotline 0800-HELP.
Use ONLY retrieved runbooks when they match the SAME product/symptom.
Copy cause codes exactly (LIFT-MON-1, BREW-PDF-7, WIFI-BALANCE-42).
If nothing matches, reply: MISS — no matching runbook in the operator KB. Escalate to Level 2.
Keep answers under 100 words."""


def ask(question: str) -> None:
    payload = {
        "model": llm_id,
        "input": question,
        "instructions": SYSTEM,
        "temperature": 0.1,
        "tools": [{"type": "file_search", "vector_store_ids": [vs_id]}],
    }
    r = requests.post(f"{V1}/responses", json=payload, timeout=180)
    r.raise_for_status()
    body = r.json()
    print("Q:", question)
    for item in body.get("output") or []:
        if item.get("type") == "file_search_call":
            results = item.get("results") or []
            print(f"  retrieved: {len(results)}")
            for res in results[:3]:
                preview = (res.get("text") or "").replace("\n", " ")[:90]
                print(f"   - score={res.get('score'):.2f} {preview}")
        if item.get("type") == "message":
            for c in item.get("content") or []:
                if c.get("type") == "output_text":
                    print("A:", c.get("text"))
    print()


ask("The elevator refuses Mondays")
ask("My fridge is singing opera")

### Success
- Elevator → retrieved chunk mentions `LIFT-MON-1` / *Maintenance spirits*
- Fridge → MISS (or clear refusal), even if a weak nearest chunk appears

Save `vs_id` if you want a Streamlit app to call the same OGX `/v1/responses` endpoint later.

**Milvus:** same ingest/query pattern with `provider_id: "milvus-remote"` after you install Milvus + etcd (official vector-DB chapter). This lab installs **dedicated pgvector**.